In [1]:
import json
import re
from itertools import product
from pathlib import Path

import pandas as pd

RAW = Path("data/raw")
INTERIM = Path("data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)
SEED = 42

with open(RAW / "simidioms" / "sim_idioms.jsonl", encoding="utf-8") as f:
    sim = [json.loads(line) for line in f]

# ядро: кластери, де є приклади і українською, і англійською
core = [r for r in sim if r["uk"].get("example") and r["en"].get("example")]
len(sim), len(core)

(2262, 309)

In [2]:
def texts(items):
    return [x["text"].strip() for x in items or [] if x.get("text", "").strip()]


def idiom_variants(side):
    items = side.get("idiom", [])
    own = texts([x for x in items if x.get("field") == "idiom"])
    return own or texts(items)


def meaning(r, lang):
    other = "en" if lang == "uk" else "uk"
    for side, key in [(lang, "figurative_meaning"),
                      (lang, "figurative_meaning_translated"),
                      (other, "figurative_meaning")]:
        m = texts(r[side].get(key))
        if m:
            return m[0], f"{side}.{key}"
    return None, None


rows = []
for r in core:
    for lang in ["uk", "en"]:
        m, src = meaning(r, lang)
        rows.append({
            "cluster_id": r["id_"],
            "language": lang,
            "variants": idiom_variants(r[lang]),
            "meaning": m,
            "meaning_source": src,
            "examples": texts(r[lang].get("example")),
        })
clusters = pd.DataFrame(rows)
clusters.head()

,cluster_id,language,variants,meaning,meaning_source,examples
0,SIM-0,uk,[В повному розпалі],Максимальна потужність або зусилля.,uk.figurative_meaning_translated,[Будівництво Дніпрогесу в повному розпалі.]
1,SIM-0,en,"[Go full blast, full blast]",Maximum capacity or effort.,en.figurative_meaning,[At Fallgate a short branch leading to a quarr...
2,SIM-100,uk,[Всякий мотлох],Різноманітні дрібнички.,uk.figurative_meaning_translated,"[Наче й невдобно сипати в рів, бо багато всяко..."
3,SIM-100,en,[odds and ends],Miscellaneous items.,en.figurative_meaning,"[I scraped my palette, stuck my brushes in a b..."
4,SIM-1000,uk,[Як на гріх],Погіршити і без того складну ситуацію.,uk.figurative_meaning_translated,"[Ніколи сього зроду не було, а се, як на гріх...."


In [3]:
clusters.groupby("language").agg(
    n=("cluster_id", "size"),
    with_meaning=("meaning", lambda s: s.notna().sum()),
    n_examples=("examples", lambda s: s.str.len().sum()),
)

,n,with_meaning,n_examples
language,,,
en,309,309,1100
uk,309,309,1155


In [4]:
clusters[clusters.language == "uk"].variants.explode().sample(15, random_state=SEED)

610                   Віддати душу Богові
318                         Казати правду
316                             Хочеш вір
502                               З голки
92                       Вилетіти в трубу
458               ЗДАВАТИ / ЗДАТИ ПОЗИЦІЇ
304                    Захопити зненацька
170                        Втікати щодуху
358    БАБА (ЩЕ) НАДВОЄ ВОРОЖИЛА (гадала)
328                  Спати без задніх ніг
322                     Вибитися із колії
496                       Легко відбутися
462                          Гидке каченя
98                  Триматися на волосині
334               риймати щось за належне
Name: variants, dtype: str

In [5]:
PLACEHOLDERS = {
    "uk": {"кого", "кому", "кім", "ким", "чого", "чому", "чим", "що", "чім", "хто",
           "когось", "чогось", "щось",
           "кого-небудь", "кому-небудь", "ким-небудь", "чого-небудь", "чому-небудь",
           "чим-небудь", "що-небудь", "хто-небудь", "який-небудь",
           "кого-чого", "кому-чому", "ким-чим"},
    "en": {"one's", "someone's", "somebody's", "someone", "somebody", "something",
           "sth", "sb", "one", "oneself", "yourself", "it"},
}


def clean_entry(entry, lang):
    s = entry.replace("’", "'").replace("ʼ", "'")
    s = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", s)   # необов'язкові частини
    s = re.sub(r"\.\.\.|…", " ", s)

    words = s.split()
    letters = [c for c in s if c.isalpha()]
    if letters and sum(c.isupper() for c in letters) > 0.6 * len(letters):
        # запис ВЕЛИКИМИ: слова з малих літер = керування, викидаємо
        words = [w for w in words if not any(c.islower() for c in w)]

    s = re.sub(r"\s*/\s*", "/", " ".join(words).lower())   # "a / b" -> "a/b"

    out = []
    for combo in product(*[w.split("/") for w in s.split()]):
        ws = [re.sub(r"[^\w'-]", "", w).strip("-'") for w in combo]
        ws = [w for w in ws if w and w not in PLACEHOLDERS[lang]]
        if ws:
            out.append(" ".join(ws))
    return list(dict.fromkeys(out))   # без дублікатів, порядок зберігається

In [6]:
tests = [("ПРОМОЧИТИ / ПРОМОЧУВАТИ ГОРЛО (гортань)", "uk"),
         ("БАБА (ЩЕ) НАДВОЄ ВОРОЖИЛА (гадала)", "uk"),
         ("риймати щось за належне", "uk"),
         ("Через пень-колоду", "uk"),
         ("keep one's head above water", "en")]
for e, lang in tests:
    print(f"{e!r:45} -> {clean_entry(e, lang)}")

'ПРОМОЧИТИ / ПРОМОЧУВАТИ ГОРЛО (гортань)'     -> ['промочити горло', 'промочувати горло']
'БАБА (ЩЕ) НАДВОЄ ВОРОЖИЛА (гадала)'          -> ['баба надвоє ворожила']
'риймати щось за належне'                     -> ['риймати за належне']
'Через пень-колоду'                           -> ['через пень-колоду']
"keep one's head above water"                 -> ['keep head above water']


In [8]:
clusters["clean"] = clusters.apply(
    lambda r: [c for v in r.variants for c in clean_entry(v, r.language)], axis=1)
clusters[["language", "variants", "clean"]].sample(10, random_state=SEED)

,language,variants,clean
49,en,"[play by ear, By ear]","[play by ear, by ear]"
582,uk,"[Сіль землі, Земля тримається на таких людях]","[сіль землі, земля тримається на таких людях]"
82,uk,"[Пройти сувору школу, Пройти крізь вогонь, вод...","[пройти сувору школу, пройти крізь вогонь воду..."
305,en,[catch napping],[catch napping]
109,en,"[forgive and forget, Forget and forgive]","[forgive and forget, forget and forgive]"
606,uk,[Чухати потилицю],[чухати потилицю]
268,uk,[Ні за яких обставин],[ні за яких обставин]
353,en,"[after all, in the long run, long run, when al...","[after all, in the long run, long run, when al..."
328,uk,"[Спати мертвим сном, Спати без задніх ніг]","[спати мертвим сном, спати без задніх ніг]"
533,en,[Tempt fate],[tempt fate]
